**Mini Project**

**Title: Analyse a Real-World Dataset, Build an Interactive Dashboard and Present Actionable Insights with Ethical Considerations**

**Enrollment No.: 240840116063**

**Aim:
To analyse a real-world business dataset (retail sales), design and develop an interactive dashboard using Dash/Plotly for business reporting, and present actionable insights while addressing ethical considerations.**

**Step 1: Install libraries**

In [1]:
# Install the required libraries
!pip install -q dash plotly pandas numpy

**Step 2: Import Required libraries**

In [2]:
# Import Pandas and NumPy
import pandas as pd
import numpy as np
# Import Plotly Express
import plotly.express as px
# Import Dash components
from dash import Dash, dcc, html, Input, Output

# Display settings for tables
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

**Step 3: Create the dataset**

In [3]:
# Create a simulated retail sales dataset (India, year 2024)
np.random.seed(42)
n = 400

# Product catalogue: (category, unit price in INR, base profit margin)
catalogue = {
    "Laptop":         ("Technology", 55000, 0.14),
    "Smartphone":     ("Technology", 28000, 0.12),
    "Tablet":         ("Technology", 22000, 0.13),
    "Headphones":     ("Technology", 3500, 0.25),
    "Office Chair":   ("Furniture", 7500, 0.18),
    "Desk":           ("Furniture", 12000, 0.16),
    "Bookcase":       ("Furniture", 9000, 0.15),
    "Notebook Pack":  ("Office Supplies", 450, 0.30),
    "Printer Paper":  ("Office Supplies", 650, 0.22),
    "Stationery Set": ("Office Supplies", 900, 0.35),
}
states = {
    "North": ["Delhi", "Punjab"],
    "South": ["Karnataka", "Tamil Nadu"],
    "East":  ["West Bengal", "Odisha"],
    "West":  ["Gujarat", "Maharashtra"],
}

region = np.random.choice(list(states), n, p=[0.24, 0.28, 0.16, 0.32])
state = [np.random.choice(states[r]) for r in region]
product = np.random.choice(list(catalogue), n,
          p=[0.10, 0.14, 0.08, 0.12, 0.08, 0.06, 0.06, 0.14, 0.12, 0.10])
category = [catalogue[p][0] for p in product]
price = np.array([catalogue[p][1] for p in product])
margin = np.array([catalogue[p][2] for p in product])

# Festive season (Oct-Nov) receives more orders
month_w = np.array([6, 6, 7, 7, 7, 7, 7, 8, 9, 12, 14, 10], dtype=float)
month = np.random.choice(np.arange(1, 13), n, p=month_w / month_w.sum())
day = np.random.randint(1, 28, n)
order_date = pd.to_datetime(dict(year=2024, month=month, day=day))

quantity = np.where(price > 10000, np.random.randint(1, 4, n),
                    np.random.randint(1, 6, n))
discount = np.random.choice([0, 0.05, 0.10, 0.20, 0.30], n,
                            p=[0.30, 0.25, 0.20, 0.15, 0.10])
sales = (quantity * price * (1 - discount)).round(0)
profit = (sales * (margin - 0.6 * discount
                   + np.random.normal(0, 0.03, n))).round(0)

df = pd.DataFrame({
    "Order_ID": [f"ORD{1000 + i}" for i in range(n)],
    "Order_Date": order_date,
    "Customer_ID": [f"C{np.random.randint(1, 181):04d}" for _ in range(n)],
    "Segment": np.random.choice(["Consumer", "Corporate", "Small Business"],
                                n, p=[0.5, 0.3, 0.2]),
    "Region": region,
    "State": state,
    "Category": category,
    "Product": product,
    "Quantity": quantity,
    "Discount": discount,
    "Sales": sales.astype(int),
    "Profit": profit.astype(int),
}).sort_values("Order_Date").reset_index(drop=True)
df["Order_ID"] = [f"ORD{1000 + i}" for i in range(n)]

# Simulate real-world data-quality issues (missing values and duplicates)
df.loc[[15, 88, 140, 222, 301, 377], "Segment"] = np.nan
df = pd.concat([df, df.iloc[[10, 50, 200, 350]]], ignore_index=True)

print(df.head(10))
print("\nShape:", df.shape)

  Order_ID Order_Date Customer_ID         Segment Region        State         Category        Product  Quantity  Discount   Sales  Profit
0  ORD1000 2024-01-01       C0005        Consumer  North        Delhi        Furniture           Desk         2      0.20   19200    1055
1  ORD1001 2024-01-01       C0165        Consumer   East  West Bengal        Furniture       Bookcase         3      0.05   25650    4261
2  ORD1002 2024-01-02       C0166       Corporate   East       Odisha  Office Supplies  Notebook Pack         5      0.05    2138     616
3  ORD1003 2024-01-04       C0023  Small Business  North        Delhi        Furniture           Desk         2      0.00   24000    3967
4  ORD1004 2024-01-05       C0008        Consumer  North        Delhi       Technology     Smartphone         3      0.20   67200   -3069
5  ORD1005 2024-01-06       C0129        Consumer   West      Gujarat  Office Supplies  Printer Paper         1      0.05     618     105
6  ORD1006 2024-01-06       C0052 

**Step 4: Inspect the dataset**

In [4]:
# Inspect the structure of the dataset
df.info()
print()
# Statistical summary of numeric columns
print(df[["Quantity", "Discount", "Sales", "Profit"]].describe().round(2))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Order_ID     404 non-null    object        
 1   Order_Date   404 non-null    datetime64[ns]
 2   Customer_ID  404 non-null    object        
 3   Segment      398 non-null    object        
 4   Region       404 non-null    object        
 5   State        404 non-null    object        
 6   Category     404 non-null    object        
 7   Product      404 non-null    object        
 8   Quantity     404 non-null    int64         
 9   Discount     404 non-null    float64       
 10  Sales        404 non-null    int64         
 11  Profit       404 non-null    int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(7)
memory usage: 38.0+ KB

       Quantity  Discount      Sales    Profit
count    404.00    404.00     404.00    404.00
mean       2.57      0.09   2658

**Step 5: Clean and prepare the data**

In [5]:
# Check data quality issues
print("Missing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# Clean the dataset
df = df.drop_duplicates().copy()                  # remove duplicate rows
df["Segment"] = df["Segment"].fillna("Unknown")   # keep rows, label missing segment

# Create new analysis columns
df["Month_Num"] = df["Order_Date"].dt.month
df["Month"] = df["Order_Date"].dt.strftime("%b")
df["Discount_%"] = (df["Discount"] * 100).astype(int)

print("\nAfter cleaning -> shape:", df.shape)
print("Missing values left:", df.isnull().sum().sum())

Missing values:
Order_ID       0
Order_Date     0
Customer_ID    0
Segment        6
Region         0
State          0
Category       0
Product        0
Quantity       0
Discount       0
Sales          0
Profit         0
dtype: int64

Duplicate rows: 4

After cleaning -> shape: (400, 15)
Missing values left: 0


**Step 6: Calculate key performance indicators (KPIs)**

In [6]:
# Calculate key performance indicators (KPIs)
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()

print(f"Total Sales        : ₹{total_sales:,.0f}")
print(f"Total Profit       : ₹{total_profit:,.0f}")
print(f"Profit Margin      : {total_profit / total_sales * 100:.2f}%")
print(f"Total Orders       : {df['Order_ID'].nunique()}")
print(f"Average Order Value: ₹{df['Sales'].mean():,.0f}")
print(f"Loss-making Orders : {(df['Profit'] < 0).sum()}")

Total Sales        : ₹10,577,484
Total Profit       : ₹1,002,160
Profit Margin      : 9.47%
Total Orders       : 400
Average Order Value: ₹26,444
Loss-making Orders : 31


**Step 7: Exploratory data analysis**

In [7]:
# Create a reusable summary function
def summarise(col):
    t = df.groupby(col)[["Sales", "Profit"]].sum()
    t["Margin_%"] = (t["Profit"] / t["Sales"] * 100).round(2)
    return t

print("--- By Region ---")
print(summarise("Region").sort_values("Profit", ascending=False))

print("\n--- By Category ---")
print(summarise("Category").sort_values("Profit", ascending=False))

print("\n--- By Product ---")
print(summarise("Product").sort_values("Profit", ascending=False))

print("\n--- By Discount (%) ---")
print(summarise("Discount_%").sort_index())

--- By Region ---
          Sales  Profit  Margin_%
Region                           
West    3197442  313229      9.80
South   2776461  270324      9.74
North   2496450  254562     10.20
East    2107131  164045      7.79

--- By Category ---
                   Sales  Profit  Margin_%
Category                                  
Technology       8559300  752933      8.80
Furniture        1798500  196085     10.90
Office Supplies   219684   53142     24.19

--- By Product ---
                  Sales  Profit  Margin_%
Product                                  
Laptop          3421000  297759      8.70
Smartphone      3243800  232605      7.17
Tablet          1377200  112703      8.18
Headphones       517300  109866     21.24
Desk             580800   69426     11.95
Office Chair     551250   64758     11.75
Bookcase         666450   61901      9.29
Stationery Set    79650   24189     30.37
Notebook Pack     66062   16804     25.44
Printer Paper     73972   12149     16.42

--- By Discount (

In [8]:
# Check the relationship between numeric columns
print(df[["Quantity", "Discount", "Sales", "Profit"]].corr().round(2))

          Quantity  Discount  Sales  Profit
Quantity      1.00      0.11  -0.08    0.03
Discount      0.11      1.00  -0.11   -0.41
Sales        -0.08     -0.11   1.00    0.77
Profit        0.03     -0.41   0.77    1.00


**Step 8: Visual exploration with Plotly**

In [9]:
# Monthly sales and profit trend
monthly = (df.groupby(["Month_Num", "Month"], as_index=False)[["Sales", "Profit"]]
             .sum().sort_values("Month_Num"))

fig = px.line(monthly, x="Month", y=["Sales", "Profit"], markers=True,
              title="Monthly Sales and Profit Trend (2024)")
fig.show()

In [10]:
# Impact of discount on profit
disc = df.groupby("Discount_%", as_index=False)["Profit"].sum()
disc["Discount_%"] = disc["Discount_%"].astype(str) + "%"

fig = px.bar(disc, x="Discount_%", y="Profit", color="Profit",
             color_continuous_scale="RdYlGn",
             title="Total Profit by Discount Level")
fig.show()

**Step 9: Create the dashboard layout**

In [11]:
# Helper function to build a KPI card
def kpi_card(title, value, color):
    return html.Div([
        html.P(title, style={"margin": "0", "fontSize": "14px", "color": "#555"}),
        html.H2(value, style={"margin": "5px 0 0 0", "color": color}),
    ], style={"flex": "1", "padding": "15px", "borderRadius": "8px",
              "backgroundColor": "#f4f6f9", "borderLeft": f"6px solid {color}"})

# Create a Dash application
app = Dash(__name__)

# Design the dashboard
app.layout = html.Div([

    # Add title
    html.H1(
        "Business Reporting Dashboard",
        style={"textAlign": "center", "color": "blue"}
    ),

    # Create the filters (two dropdowns)
    html.Div([
        html.Div([
            html.Label("Select Region:"),
            dcc.Dropdown(
                id="region",
                options=[{"label": "All Regions", "value": "All"}] + [
                    {"label": r, "value": r}
                    for r in sorted(df["Region"].unique())
                ],
                value="All",
                clearable=False
            ),
        ], style={"width": "48%", "display": "inline-block"}),

        html.Div([
            html.Label("Select Category:"),
            dcc.Dropdown(
                id="category",
                options=[{"label": "All Categories", "value": "All"}] + [
                    {"label": c, "value": c}
                    for c in sorted(df["Category"].unique())
                ],
                value="All",
                clearable=False
            ),
        ], style={"width": "48%", "display": "inline-block",
                  "marginLeft": "4%"}),
    ]),

    # Area for KPI cards
    html.Div(id="kpi_cards",
             style={"display": "flex", "gap": "15px", "margin": "20px 0"}),

    # Add charts
    dcc.Graph(id="trend_chart"),
    dcc.Graph(id="product_chart"),
    dcc.Graph(id="discount_chart"),
    dcc.Graph(id="segment_chart"),

    # Ethical note for the viewer
    html.P("Note: Data is simulated. No personal information is used. "
           "Insights support, but do not replace, human decisions.",
           style={"fontSize": "12px", "color": "gray"})

], style={"padding": "20px"})

**Step 10: Add interactive dashboard functionality**

In [12]:
# Update the dashboard
@app.callback(
    Output("kpi_cards", "children"),
    Output("trend_chart", "figure"),
    Output("product_chart", "figure"),
    Output("discount_chart", "figure"),
    Output("segment_chart", "figure"),
    Input("region", "value"),
    Input("category", "value")
)

# Define the function
def update_dashboard(region, category):

    # Filter the dataset
    filtered = df.copy()
    if region != "All":
        filtered = filtered[filtered["Region"] == region]
    if category != "All":
        filtered = filtered[filtered["Category"] == category]

    # Calculate KPIs
    total_sales = filtered["Sales"].sum()
    total_profit = filtered["Profit"].sum()
    margin = total_profit / total_sales * 100 if total_sales else 0

    cards = [
        kpi_card("Total Sales", f"₹{total_sales:,.0f}", "#636EFA"),
        kpi_card("Total Profit", f"₹{total_profit:,.0f}", "#00CC96"),
        kpi_card("Profit Margin", f"{margin:.1f}%", "#AB63FA"),
        kpi_card("Total Orders", f"{len(filtered)}", "#FFA15A"),
        kpi_card("Avg Order Value", f"₹{filtered['Sales'].mean():,.0f}", "#EF553B"),
    ]

    # Create a line chart (monthly trend)
    monthly = (filtered.groupby(["Month_Num", "Month"], as_index=False)
                       [["Sales", "Profit"]].sum().sort_values("Month_Num"))
    trend_fig = px.line(monthly, x="Month", y=["Sales", "Profit"],
                        markers=True, title="Monthly Sales and Profit Trend")

    # Create a horizontal bar chart (profit by product)
    prod = (filtered.groupby(["Product", "Category"], as_index=False)["Profit"]
                    .sum().sort_values("Profit"))
    product_fig = px.bar(prod, x="Profit", y="Product", color="Category",
                         orientation="h", title="Profit by Product")

    # Create a bar chart (profit by discount level)
    disc = filtered.groupby("Discount_%", as_index=False)["Profit"].sum()
    disc["Discount_%"] = disc["Discount_%"].astype(str) + "%"
    discount_fig = px.bar(disc, x="Discount_%", y="Profit", color="Profit",
                          color_continuous_scale="RdYlGn",
                          title="Profit by Discount Level")

    # Create a donut chart (sales share by customer segment)
    segment_fig = px.pie(filtered, names="Segment", values="Sales", hole=0.4,
                         title="Sales Share by Customer Segment")

    return cards, trend_fig, product_fig, discount_fig, segment_fig

**Step 11: Run the dashboard in Google Colab**

In [13]:
# Run the dashboard
app.run(jupyter_mode="inline", debug=False, port=8050)



<IPython.core.display.Javascript object>